<a href="https://colab.research.google.com/github/asierra383/ScamBusters_Agent/blob/main/Scam_Busters_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

START

In [1]:
pip install google-adk

In [2]:
#Import ADK components
import gspread
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters
from google.adk.memory import InMemoryMemoryService
from google.adk.tools import preload_memory
import google.genai as genai

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [3]:
import os
from google.colab import userdata
api_key = userdata.get('GOOGLE_API_KEY')
os.environ['GOOGLE_API_KEY'] = api_key

In [4]:
#Configure Retry options
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)

In [5]:
# 1. Finds the main HYIP monitor sites
discovery_agent = LlmAgent(
    name="DiscoveryAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        api_key=api_key,
        retry_options=retry_config
    ),
    instruction="Find current HYIP monitoring websites. List only the URLs.",
    tools=[google_search],
    output_key="hyip_monitor_list"
)

# 2. Next agent to browse the sites advertised in HYIP monitoring websites and extract content
browser_agent = LlmAgent(
    name="BrowserAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        api_key=api_key,
        retry_options=retry_config),
    instruction="""Given the list of HYIP monitoring websites in {hyip_monitor_list},
    use Google Search to visit each website and extract the cryptocurrency websites it has linked.
    Combine all extracted websites into a single comprehensive string and make
    this combined content available as 'site_content' for the next agent.""",
    tools=[google_search],
    output_key="site_content"
)

print("✅ Discovery Agent and Browser Agent defined.")

✅ Discovery Agent and Browser Agent defined.


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# --- STEP 1: SPREADSHEET SETUP ---
import gspread
gc = gspread.service_account(filename='/content/drive/My Drive/Colab Notebooks/scambusters.json')
sh = gc.open("ScamBusters_Tracker")
worksheet = sh.get_worksheet(0)

# --- STEP 2: DEFINE THE TOOL AS A FUNCTION ---
def add_to_spreadsheet(url: str, category: str, notes: str):
    """
    Appends a new website's details to the tracking spreadsheet.
    Use this whenever a new relevant website is discovered.

    Args:
        url (str): The URL of the website to add.
        category (str): The category for the website (e.g., 'HYIP Monitor').
        notes (str): Any additional notes about the website.
    """
    try:
        # Check for duplicates before adding
        existing_urls = worksheet.col_values(1)
        if url in existing_urls:
            return f"Skipped: {url} is already in the sheet."

        worksheet.append_row([url, category, notes])
        return f"Successfully added {url} to the spreadsheet."
    except Exception as e:
        return f"Error updating spreadsheet: {str(e)}"

print("✅ Spreadsheet function defined.")

✅ Spreadsheet function defined.


In [8]:
# 3. Processes site content and logs new URLs to the spreadsheet
spreadsheet_agent = LlmAgent(
    name="SpreadsheetAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        api_key=api_key,
        retry_options=retry_config
    ),
    instruction="""Analyze the content in {site_content}.
    Identify new, unique HYIP related URLs mentioned in the content.
    For each new unique URL found, use the 'add_to_spreadsheet' tool with category 'HYIP Monitor' and notes 'Discovered by agent'.
    Pass the original {site_content} along to the next agent unchanged.""",
    tools=[add_to_spreadsheet],
    output_key="site_content"
)

print("✅ Spreadsheet Agent defined.")

✅ Spreadsheet Agent defined.


In [9]:
# 4. Analyzes the extracted text for your specific scam keywords
analyst_agent = LlmAgent(
    name="AnalystAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
    retry_options=retry_config),
    instruction="""Analyze the content in {site_content}.
    Search for scam markers like 'guaranteed high returns' or anonymous teams.
    Provide a final risk assessment.""",
    output_key="final_scam_report"
)

print("✅ Analyst Agent defined.")

✅ Analyst Agent defined.


In [10]:
# Re-initialize the pipeline with updated agents
scam_pipeline = SequentialAgent(
    name="ScamDetectionPipeline",
    sub_agents=[discovery_agent, browser_agent, spreadsheet_agent, analyst_agent]
)

print("✅ Pipeline updated with agents.")

✅ Pipeline updated with agents.


In [11]:
#Run the multi-agent system
runner = InMemoryRunner(agent=scam_pipeline)

print("✅ Runner updated.")

✅ Runner updated.


In [12]:
#Run by prompting for answer
response = await runner.run_debug(
    "What hyip monitoring websites have recently updated their lists within the past day?"
)


 ### Created new session: debug_session_id

User > What hyip monitoring websites have recently updated their lists within the past day?
DiscoveryAgent > Several HYIP monitoring websites appear to have updated their lists recently, with some showing activity within the past day.

Here are some URLs that indicate recent updates:

*   All HYIP Monitors .com ()
*   Allmonitors24.com ()
*   CryptoHyip.net ()
*   H-metrics.com ()
*   HYIP.BIZ ()
*   HYIP Monitor ()
*   HYIPexplorer ()
*   upayhyip.com ()
BrowserAgent > Here are the cryptocurrency websites linked by the HYIP monitoring websites you listed:

*   **All HYIP Monitors .com**: This site links to various HYIP programs that may involve cryptocurrencies. Examples include app.tensorhive.io, moneprix.com, primetrade.cc, xolonetwork.net, simplecryptohub.com, and ramonainv.com. It also lists scam programs like cryptobotics.net and solforge.biz.

*   **Allmonitors24.com**: This website lists HYIP programs and also mentions cryptocurrency

SpreadsheetAgent > The following are new, unique HYIP related URLs:
app.tensorhive.io, moneprix.com, primetrade.cc, xolonetwork.net, simplecryptohub.com, ramonainv.com, cryptobotics.net, solforge.biz, LocalCoinSwap, Bitget, Gate.com, KuCoin, MEXC, Binance, Upbit, OKX, Bybit, HTX, Bitunix, Crypto.com, KoalaFaucet, ClxAward, CryptoCollect, CoinpayuFree, Monster Faucet, FreeZeroCo.in, DigimonBTC, MyZeroLand, NexFaucet, Marsses, Quantara, Elementex, Sixty5040, Vigo Gold, Uniminepool, Moneprix, crypto-btc.com, crypto-hyiper.com, bitcoin-trading.info, Cryptoize Limited, King Hectares, Allocra, Best-dep.com, Circuito-venture.com, Hashranchogpu.com, Bliss-p2p.com

SpreadsheetAgent > I have added all the newly discovered HYIP related URLs to the spreadsheet.
AnalystAgent > I have analyzed the content you provided, focusing on the URLs that were added to the spreadsheet. My analysis for scam markers like 'guaranteed high returns' or anonymous teams is limited because the provided text primarily 